# Wellbore Geology Prediction — Full Resolution (LightGBM)
Clones source from GitHub, downloads data from Kaggle, trains on all 773 wells.
Google Drive model caching avoids retraining on repeated runs.

Uses LightGBM with DTW/cross-correlation typewell features, Savitzky-Golay smoothing, and ONNX export.

**Before running:**
1. Get a Kaggle API token at https://www.kaggle.com/settings → API → Generate New Token
2. Set it as a Colab secret: click the 🔑 key icon in the left panel → add `KAGGLE_API_TOKEN`

In [ ]:
# --- SET THESE ---
GITHUB_REPO = "https://github.com/adjanour/rogii-wellbore-geology-prediction.git"
# -----------------

import os, sys
if not os.path.exists('/content/rogii-wellbore-geology-prediction/src'):
    !git clone $GITHUB_REPO /content/rogii-wellbore-geology-prediction
else:
    print('Source already cloned')

sys.path.insert(0, '/content/rogii-wellbore-geology-prediction')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CACHE_DIR = '/content/drive/MyDrive/wellbore_model'
os.makedirs(CACHE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(CACHE_DIR, 'lgb_model.txt')
FEATURES_PATH = os.path.join(CACHE_DIR, 'feature_names.txt')

In [ ]:
from google.colab import userdata

data_dir = '/content/rogii-wellbore-geology-prediction/train'
if not os.path.exists(data_dir) or len(os.listdir(data_dir)) < 10:
    token = userdata.get('KAGGLE_API_TOKEN')
    os.environ['KAGGLE_API_TOKEN'] = token
    !kaggle competitions download rogii-wellbore-geology-prediction
    !unzip -q rogii-wellbore-geology-prediction.zip -d /content/rogii-wellbore-geology-prediction
    !rm rogii-wellbore-geology-prediction.zip
    print('Data ready')
else:
    print('Data already present')

In [ ]:
!pip install -q lightgbm dtaidistance scikit-learn pandas numpy scipy matplotlib

In [ ]:
import gc, numpy as np, pandas as pd, lightgbm as lgb
from pathlib import Path
from scipy.signal import savgol_filter

from src.data.loader import load_all_wells, load_horizontal, load_typewell
from src.features.build_features import build_features, FEATURE_COLS, smooth_tvt

PROJ = Path('/content/rogii-wellbore-geology-prediction')
OUTPUT = '/content/submission.csv'

In [ ]:
if os.path.exists(MODEL_PATH):
    print('Loading cached model from Drive...')
    model = lgb.LGBMRegressor()
    model = model.load_model(MODEL_PATH)
    with open(FEATURES_PATH) as f:
        avail = f.read().strip().split(',')
    print(f'Loaded model ({len(avail)} features)')
else:
    print('Training LightGBM from scratch on all 773 wells...')
    wells = load_all_wells(PROJ, is_train=True)
    all_ids = list(wells.keys())

    X_list, y_list = [], []
    for wid in all_ids:
        hw, tw = wells[wid]
        df = build_features(hw, tw, is_train=True)
        avail = [c for c in FEATURE_COLS if c in df.columns]
        X_list.append(df[avail].values.astype(np.float32))
        y_list.append(df['TVT'].values.astype(np.float32))
        del df, hw, tw; gc.collect()

    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list)
    print(f'Training: {X.shape[0]} rows, {X.shape[1]} features')

    md_idx = FEATURE_COLS.index('md') if 'md' in FEATURE_COLS else -1
    mono = [0] * len(FEATURE_COLS)
    if md_idx >= 0:
        mono[md_idx] = 1

    model = lgb.LGBMRegressor(
        objective='regression_l1', metric='rmse',
        n_estimators=2000, learning_rate=0.03,
        num_leaves=63, max_depth=-1, min_child_samples=20,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        monotone_constraints=mono,
        n_jobs=-1, random_state=42, verbose=-1,
    )
    model.fit(X, y)

    model.booster_.save_model(MODEL_PATH)
    with open(FEATURES_PATH, 'w') as f:
        f.write(','.join(avail))
    print(f'Cached to drive at {MODEL_PATH}')

In [ ]:
TEST_DIR = PROJ / 'test'
well_ids = sorted(set(p.stem.split('__')[0] for p in TEST_DIR.glob('*__horizontal_well.csv')))
all_out_ids, all_tvts = [], []

for wid in well_ids:
    hw = load_horizontal(TEST_DIR, wid, is_train=False)
    tw = load_typewell(TEST_DIR, wid, is_train=False)
    nan_mask = hw['TVT_input'].isna().values
    df = build_features(hw, tw, is_train=False)
    avail_cols = [c for c in FEATURE_COLS if c in df.columns]
    preds = smooth_tvt(model.predict(df[avail_cols].values.astype(np.float32)))
    for idx in np.where(nan_mask)[0]:
        all_out_ids.append(f'{wid}_{idx}')
        all_tvts.append(preds[idx])
    print(f'  {wid}: done')
    del hw, tw, df, preds; gc.collect()

sub = pd.DataFrame({'id': all_out_ids, 'tvt': all_tvts})
sub.to_csv(OUTPUT, index=False)
print(f'Saved {len(sub)} predictions')

from google.colab import files
files.download(OUTPUT)